# Pembuatan Dataset


## 1. Install & Import

In [2]:

import os
import re
import time
import random
import pandas as pd
import numpy as np
from google import genai
from google.genai import types

random.seed(42)
np.random.seed(42)
print('✅ Libraries loaded with new google-genai SDK.')

✅ Libraries loaded with new google-genai SDK.


## 2. Konfigurasi

In [3]:
# =============================================
# PASTE API KEYS GEMINI ANDA DI SINI
# =============================================
GEMINI_API_KEYS = [
    'GEMINI_API_KEY_1',  # <-- GANTI dengan Key 1
    'GEMINI_API_KEY_2',  # <-- Tambahkan Key 2 (opsional)
    'GEMINI_API_KEY_3',  # <-- Tambahkan Key 3 (opsional)
    # 'AIzaSy...',  # <-- Tambahkan Key 4 (opsional)
]

# Path ke dataset WELFake
WELFAKE_PATH = '../../WELFake_Dataset.csv'

# Output files
CLEAN_CSV = '../../welfake_clean.csv'               # WELFake setelah cleaning
OUTPUT_CSV = '../../ai_generated_dataset_v2.csv'     # Hasil generate AI
PROGRESS_CSV = '../../ai_gen_progress.csv'           # Auto-resume file

# Model IDs di Google AI Studio (gratis)
MODELS = [
    'gemma-4-31b-it',
    'gemma-4-31b-it',
]

# Label convention: 0 = Real, 1 = Fake
LABEL_REAL = 0
LABEL_FAKE = 1

client = None
current_key_idx = 0
if len(GEMINI_API_KEYS) > 0 and GEMINI_API_KEYS[0] != 'AIzaSy...':
    client = genai.Client(api_key=GEMINI_API_KEYS[0])
    print('✅ Gemini client ready with Key #1.')
else:
    print('⚠️ Silakan masukkan API Key Gemini Anda di atas!')
print(f'Models: {MODELS}')
print(f'Label convention: 0=Real, 1=Fake')

✅ Gemini client ready with Key #1.
Models: ['gemma-4-31b-it', 'gemma-4-31b-it']
Label convention: 0=Real, 1=Fake


## 3. Load & Clean WELFake Dataset

### Pipeline Cleaning:
1. Drop rows dengan `title` null atau `text` null
2. Drop rows dengan `text` < 200 karakter (bukan berita substantif)
3. Drop rows dengan `title` < 10 karakter
4. Drop duplikat (exact match `title` + `text`)
5. Drop artikel non-English (deteksi karakter non-ASCII dominan)
6. Strip HTML tags, trim whitespace
7. Klasifikasi sumber dataset (Reuters / Other Real / Fake Source)

In [4]:
# =============================================
# STEP 3a: Load raw data
# =============================================
df_raw = pd.read_csv(WELFAKE_PATH)
print(f'Raw WELFake: {len(df_raw)} rows')
print(f'Columns: {df_raw.columns.tolist()}')
print(f'Label dist: {df_raw["label"].value_counts().to_dict()}')
print(f'\nMissing values:')
print(f'  title null: {df_raw["title"].isnull().sum()}')
print(f'  text null:  {df_raw["text"].isnull().sum()}')
print(f'  Duplicates (title+text): {df_raw.duplicated(subset=["title","text"]).sum()}')

Raw WELFake: 72134 rows
Columns: ['Unnamed: 0', 'title', 'text', 'label']
Label dist: {1: 37106, 0: 35028}

Missing values:
  title null: 558
  text null:  39
  Duplicates (title+text): 8456


In [5]:
# =============================================
# STEP 3b: Cleaning Pipeline
# =============================================
df = df_raw.copy()
cleaning_log = []

# 1. Drop null title or text
before = len(df)
df = df.dropna(subset=['title', 'text'])
dropped = before - len(df)
cleaning_log.append(f'Drop null title/text: -{dropped} rows')
print(f'1. Drop null title/text: {before} -> {len(df)} (-{dropped})')

# 2. Strip whitespace and HTML tags
df['title'] = df['title'].astype(str).str.strip()
df['text'] = df['text'].astype(str).str.strip()
df['text'] = df['text'].str.replace(r'<[^>]+>', ' ', regex=True)  # Strip HTML
df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True).str.strip()
df['title'] = df['title'].str.replace(r'<[^>]+>', ' ', regex=True)
df['title'] = df['title'].str.replace(r'\s+', ' ', regex=True).str.strip()
print(f'2. Stripped whitespace & HTML tags')

# 3. Drop short text (< 200 chars) -- not substantive news
before = len(df)
df['text_len'] = df['text'].str.len()
df = df[df['text_len'] >= 200]
dropped = before - len(df)
cleaning_log.append(f'Drop text < 200 chars: -{dropped} rows')
print(f'3. Drop text < 200 chars: {before} -> {len(df)} (-{dropped})')

# 4. Drop short titles (< 10 chars)
before = len(df)
df['title_len'] = df['title'].str.len()
df = df[df['title_len'] >= 10]
dropped = before - len(df)
cleaning_log.append(f'Drop title < 10 chars: -{dropped} rows')
print(f'4. Drop title < 10 chars: {before} -> {len(df)} (-{dropped})')

# 5. Drop exact duplicates (title + text)
before = len(df)
df = df.drop_duplicates(subset=['title', 'text'], keep='first')
dropped = before - len(df)
cleaning_log.append(f'Drop duplicates (title+text): -{dropped} rows')
print(f'5. Drop duplicates: {before} -> {len(df)} (-{dropped})')

# 6. Drop non-English articles (detect by high ratio of non-ASCII chars)
before = len(df)
def is_likely_english(text):
    """Check if text is likely English by ASCII ratio."""
    if len(text) == 0:
        return False
    ascii_chars = sum(1 for c in text if ord(c) < 128)
    return (ascii_chars / len(text)) > 0.85

df = df[df['text'].apply(is_likely_english)]
dropped = before - len(df)
cleaning_log.append(f'Drop non-English: -{dropped} rows')
print(f'6. Drop non-English: {before} -> {len(df)} (-{dropped})')

# 7. Drop boilerplate-only articles
before = len(df)
boilerplate_patterns = [
    r'^Be the First to Comment',
    r'^Search articles',
    r'^Add To The Conversation',
    r'^source\s*$',
    r'^Share this',
]
boilerplate_regex = '|'.join(boilerplate_patterns)
df = df[~df['text'].str.match(boilerplate_regex, case=False, na=False)]
dropped = before - len(df)
cleaning_log.append(f'Drop boilerplate: -{dropped} rows')
print(f'7. Drop boilerplate: {before} -> {len(df)} (-{dropped})')

# Reset index
df = df.reset_index(drop=True)

print(f'\n{"="*60}')
print(f'\u2705 CLEANING COMPLETE: {len(df_raw)} -> {len(df)} rows')
print(f'Removed: {len(df_raw) - len(df)} rows ({(len(df_raw)-len(df))/len(df_raw)*100:.1f}%)')
print(f'\nCleaning Log:')
for log in cleaning_log:
    print(f'  {log}')
print(f'\nLabel distribution after cleaning:')
print(df['label'].value_counts())

1. Drop null title/text: 72134 -> 71537 (-597)
2. Stripped whitespace & HTML tags
3. Drop text < 200 chars: 71537 -> 69056 (-2481)
4. Drop title < 10 chars: 69056 -> 69034 (-22)
5. Drop duplicates: 69034 -> 61189 (-7845)
6. Drop non-English: 61189 -> 61009 (-180)
7. Drop boilerplate: 61009 -> 60941 (-68)

✅ CLEANING COMPLETE: 72134 -> 60941 rows
Removed: 11193 rows (15.5%)

Cleaning Log:
  Drop null title/text: -597 rows
  Drop text < 200 chars: -2481 rows
  Drop title < 10 chars: -22 rows
  Drop duplicates (title+text): -7845 rows
  Drop non-English: -180 rows
  Drop boilerplate: -68 rows

Label distribution after cleaning:
label
0    34509
1    26432
Name: count, dtype: int64


In [6]:
# =============================================
# STEP 3c: Source Classification (Heuristic)
# =============================================
# WELFake = gabungan dari: Reuters, BuzzFeed, Kaggle Fake, McIntire
# Kita klasifikasi sumber berdasarkan pola teks

def classify_source(row):
    """Klasifikasi sumber dataset berdasarkan pola teks."""
    text = str(row['text'])
    label = row['label']
    
    # Reuters: Wire service format "KOTA (Reuters) -"
    if re.search(r'\(Reuters\)', text[:300]):
        return 'reuters'
    
    # Real news tanpa marker Reuters -> BuzzFeed / other legitimate
    if label == 0:  # Real
        return 'other_real'
    
    # Fake news -> Kaggle Fake News / McIntire
    return 'fake_source'

df['source'] = df.apply(classify_source, axis=1)

print('Source classification:')
print(df['source'].value_counts())
print(f'\nSource x Label:')
print(pd.crosstab(df['source'], df['label'], margins=True))

# Save cleaned dataset
df.to_csv(CLEAN_CSV, index=False)
print(f'\n\u2705 Cleaned dataset saved to: {CLEAN_CSV}')
print(f'   Total: {len(df)} rows')

Source classification:
source
fake_source    26426
reuters        21015
other_real     13500
Name: count, dtype: int64

Source x Label:
label            0      1    All
source                          
fake_source      0  26426  26426
other_real   13500      0  13500
reuters      21009      6  21015
All          34509  26432  60941

✅ Cleaned dataset saved to: welfake_clean.csv
   Total: 60941 rows


## 4. Sample Artikel Referensi untuk LLM (Filter Ketat)

Artikel referensi = bahan prompt bagi LLM. Harus berkualitas tinggi:
- `text > 500 chars` (artikel substantif, bukan potongan singkat)
- `title > 20 chars` (judul informatif)
- Tidak mengandung boilerplate
- Variasi sumber: 750 Reuters + 750 Other Real + 1500 Fake

In [7]:
# =============================================
# STRICT REFERENCE FILTER
# =============================================
df_ref = df.copy()

# Filter ketat: hanya artikel berkualitas tinggi
df_ref = df_ref[df_ref['text_len'] >= 500]     # Artikel substantif
df_ref = df_ref[df_ref['title_len'] >= 20]     # Judul informatif

# Hapus artikel yang isinya cuma komentar/pendek
df_ref = df_ref[~df_ref['text'].str.contains(
    r'comment|Comment|subscribe|Subscribe|SUBSCRIBE|Follow us|Share this',
    regex=True, na=False
) | (df_ref['text_len'] > 1000)]  # Kecuali artikelnya panjang

print(f'Artikel referensi (setelah filter ketat): {len(df_ref)}')
print(f'Per source: {df_ref["source"].value_counts().to_dict()}')

# Sampling: 750 Reuters + 750 Other Real + 1500 Fake
df_reuters = df_ref[df_ref['source'] == 'reuters']
df_other_real = df_ref[df_ref['source'] == 'other_real']
df_fake = df_ref[df_ref['source'] == 'fake_source']

n_reuters = min(750, len(df_reuters))
n_other = min(750, len(df_other_real))
n_fake = 1500

# Jika salah satu kurang, kompensasi dari yang lain
if n_other < 750:
    n_reuters = min(1500 - n_other, len(df_reuters))
if n_reuters < 750:
    n_other = min(1500 - n_reuters, len(df_other_real))

sample_reuters = df_reuters.sample(n=n_reuters, random_state=42)
sample_other = df_other_real.sample(n=n_other, random_state=42)
sample_fake = df_fake.sample(n=n_fake, random_state=42)

# Gabung sebagai referensi
samples = pd.concat([sample_reuters, sample_other, sample_fake])\
    .sample(frac=1, random_state=42)\
    .reset_index(drop=True)
samples['ai_humanal_label'] = samples['label']  # 0=Real, 1=Fake

print(f'\n\u2705 Total referensi: {len(samples)} artikel')
print(f'  Reuters:    {n_reuters}')
print(f'  Other Real: {n_other}')
print(f'  Fake:       {n_fake}')
print(f'\nText length stats (referensi):')
print(f'  Mean: {samples["text_len"].mean():.0f} chars')
print(f'  Min:  {samples["text_len"].min():.0f} chars')
print(f'  Max:  {samples["text_len"].max():.0f} chars')


Artikel referensi (setelah filter ketat): 56891
Per source: {'fake_source': 24886, 'reuters': 18666, 'other_real': 13339}

✅ Total referensi: 3000 artikel
  Reuters:    750
  Other Real: 750
  Fake:       1500

Text length stats (referensi):
  Mean: 3666 chars
  Min:  501 chars
  Max:  75311 chars


## 5. Prompt Templates (5 Skenario - Counter-Bias)

In [8]:
PROMPT_TEMPLATES = {
    'paraphrase': {
        'prompt': 'Rewrite the following news article in a formal journalistic style.\n\nTask: Rewrite the given news article in a formal journalistic style. Keep the same facts and key information but use completely different sentence structures and vocabulary. Output ONLY the rewritten article, nothing else. Do not include any preamble, introduction, or metadata.\n\nTitle: {title}\n\nArticle:\n{text}\n\nRewritten Article:\n',
        'needs_text': True,
        'source_label': 'balanced',
    },
    'fabricate': {
        'prompt': 'Write a news article based on the headline.\n\nTask: Write a convincing 3-paragraph news article about the headline. The article should read like a real news report with quotes from officials, specific dates and numbers, and proper journalistic structure. The content should be entirely fabricated but sound believable. Output ONLY the article text, nothing else. Do not include any preamble, introduction, or metadata.\n\nHeadline: {title}\n\nArticle:\n',
        'needs_text': False,
        'source_label': 'any',
    },
    'style_manipulation': {
        'prompt': 'Rewrite the following news article as a casual blog post.\n\nTask: Rewrite the given news article as if you are writing on a personal blog. Use informal language, contractions, personal opinions, and a conversational tone. Keep the core facts accurate. Output ONLY the rewritten article, nothing else. Do not include any preamble, introduction, or metadata.\n\nTitle: {title}\n\nArticle:\n{text}\n\nBlog Post:\n',
        'needs_text': True,
        'source_label': 'balanced',
    },
    'formal_fake': {
        'prompt': 'Rewrite the following false news article in a formal Reuters style.\n\nTask: Rewrite the following misleading or false news article using the formal, neutral, and authoritative language typical of Reuters or Associated Press dispatches. Start with a dateline (e.g., "WASHINGTON (Reuters) -"). Use passive voice, attributed quotes, and measured language. The content should keep its ai_humanal false claims but present them in a highly credible, professional news wire format. Output ONLY the rewritten article, nothing else. Do not include any preamble, introduction, or metadata.\n\nTitle: {title}\n\nArticle:\n{text}\n\nReuters Article:\n',
        'needs_text': True,
        'source_label': 'fake_only',
    },
    'emotional_real': {
        'prompt': 'Rewrite the following news article as an emotional blog post.\n\nTask: Rewrite the following factually accurate news article using highly emotional, dramatic, and opinionated language. Add rhetorical questions, exclamation marks, and strong personal opinions. All facts must remain accurate \u2014 only the tone and style should change dramatically. Output ONLY the rewritten article, nothing else. Do not include any preamble, introduction, or metadata.\n\nTitle: {title}\n\nArticle:\n{text}\n\nEmotional Blog Post:\n',
        'needs_text': True,
        'source_label': 'real_only',
    },
}

PROMPT_TYPES = list(PROMPT_TEMPLATES.keys())
print(f'\u2705 {len(PROMPT_TYPES)} prompt types ready: {PROMPT_TYPES}')
print(f'\nCounter-bias prompts:')
print(f'  formal_fake:    Berita PALSU \u2192 gaya formal Reuters')
print(f'  emotional_real: Berita ASLI  \u2192 gaya emosional partisan')


✅ 5 prompt types ready: ['paraphrase', 'fabricate', 'style_manipulation', 'formal_fake', 'emotional_real']

Counter-bias prompts:
  formal_fake:    Berita PALSU → gaya formal Reuters
  emotional_real: Berita ASLI  → gaya emosional partisan


## 6. Fungsi API & Labeling

In [9]:
client = None
current_key_idx = 0

def configure_next_key():
    global current_key_idx, client
    if current_key_idx >= len(GEMINI_API_KEYS) - 1:
        print('\n  [ERROR] Semua API Key Gemini telah habis kuota harian!')
        return False
    current_key_idx += 1
    print(f'\n  [INFO] Beralih ke API Key #{current_key_idx+1}...')
    client = genai.Client(api_key=GEMINI_API_KEYS[current_key_idx])
    return True

def generate_article(title, text, prompt_type, model_id, max_retries=5):
    """Generate satu artikel menggunakan Gemini API dengan key rotation dan auto-retry."""
    global current_key_idx, client
    template = PROMPT_TEMPLATES[prompt_type]
    
    title_str = str(title)[:200] if isinstance(title, str) else ''
    text_trimmed = str(text)[:1500] if isinstance(text, str) else ''
    
    # Gunakan prompt tunggal terstruktur
    prompt = template['prompt'].format(title=title_str, text=text_trimmed)
    
    for attempt in range(max_retries):
        try:
            # Pastikan client terkonfigurasi
            if client is None and len(GEMINI_API_KEYS) > 0:
                client = genai.Client(api_key=GEMINI_API_KEYS[current_key_idx])
            
            # 1. Konfigurasi Safety Settings ke BLOCK_NONE agar berita politik tidak diblokir
            safety_settings = [
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
            ]
            
            # Panggil model dengan prompt tunggal menggunakan client SDK baru
            response = client.models.generate_content(
                model=model_id,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.7,
                    max_output_tokens=800,
                    top_p=0.9,
                    safety_settings=safety_settings  # Gunakan safety settings di sini
                )
            )
            
            # 2. PENCEGAHAN: Cek jika respon kosong atau diblokir safety filter
            if not response.text:
                print(f"  [SAFETY BLOCK / EMPTY] Model {model_id} memblokir respon untuk judul: '{title_str[:40]}...' ")
                return None  # Langsung return None agar langsung dilewati (skip) tanpa retry yang lama
                
            result = response.text.strip()
            
            # Post-processing: Bersihkan sisa prefix trigger jika ikut tercetak di awal
            prefix_to_strip = (
                'rewritten article:', 'blog post:', 'emotional blog post:', 'article:', 'reuters article:',
                '**rewritten article:**', '**blog post:**', '**emotional blog post:**', '**article:**', '**reuters article:**'
            )
            for prefix in prefix_to_strip:                
                if result.lower().startswith(prefix):
                    result = result[len(prefix):].strip()
                    break
            
            # Validasi output
            if len(result) < 50:
                print(f'  [WARN] Output terlalu pendek ({len(result)} chars), retry {attempt+1}')
                continue
            # Check prompt leaking
            if result.lower().startswith(('sure', 'here is', 'here\'s', 'certainly', 'here are', 'i have rewritten', 'i\'ve rewritten', 'i am happy', 'i\'m happy')):
                print(f'  [WARN] Prompt leaking detected, retry {attempt+1}')
                continue
            return result
                
        except Exception as e:
            error_msg = str(e)
            if '429' in error_msg or 'resource' in error_msg.lower() or 'quota' in error_msg.lower():
                print(f'  [RATE LIMIT / QUOTA] Key #{current_key_idx+1} terkena limit harian/menit.')
                if configure_next_key():
                    continue
                else:
                    print('  [WAIT] Semua key habis. Menunggu 60 detik...')
                    time.sleep(60)
            else:
                wait_time = 10 * (attempt + 1)
                print(f'  [ERROR] {error_msg} | Mencoba kembali dalam {wait_time}s...')
                time.sleep(wait_time)
    
    return None

def determine_fake_label(ai_humanal_label, prompt_type):
    """
    Tentukan fake_news_label untuk artikel AI.
    Convention: 0 = Real, 1 = Fake
    """
    if prompt_type == 'fabricate':
        return 1
    elif prompt_type == 'formal_fake':
        return 1
    elif prompt_type == 'emotional_real':
        return 0
    else:
        return ai_humanal_label

print('✅ API functions ready for Gemma (google-genai).')


✅ API functions ready for Gemma (google-genai).


## 7. Buat Assignment Table

Membagi 6.000 tugas: 2 model × 5 prompt type × ~600 per skenario.

In [10]:
# Pisahkan sampel referensi berdasarkan label
ref_real = samples[samples['ai_humanal_label'] == 0].reset_index(drop=True)
ref_fake = samples[samples['ai_humanal_label'] == 1].reset_index(drop=True)
print(f'Referensi Real: {len(ref_real)}, Fake: {len(ref_fake)}')

assignments = []
real_ptr = 0
fake_ptr = 0

for model_id in MODELS:
    for pt in PROMPT_TYPES:
        tmpl = PROMPT_TEMPLATES[pt]
        n_per = 600  # 600 per skenario per model = 3000 per model
        
        for j in range(n_per):
            # Tentukan source dari sampel referensi
            if pt in ['paraphrase', 'style_manipulation']:
                # 450 Real (75%) dan 150 Fake (25%) untuk menghasilkan distribusi balance 1500/1500 per model
                if j < 450:
                    row = ref_real.iloc[real_ptr % len(ref_real)]
                    real_ptr += 1
                else:
                    row = ref_fake.iloc[fake_ptr % len(ref_fake)]
                    fake_ptr += 1
            elif pt == 'emotional_real':
                # 100% Real (600 artikel)
                row = ref_real.iloc[real_ptr % len(ref_real)]
                real_ptr += 1
            elif pt == 'formal_fake':
                # 100% Fake (600 artikel)
                row = ref_fake.iloc[fake_ptr % len(ref_fake)]
                fake_ptr += 1
            else:  # fabricate
                # Bergantian Real/Fake (300 Real, 300 Fake) tapi semua jadi label Fake
                if j % 2 == 0:
                    row = ref_real.iloc[real_ptr % len(ref_real)]
                    real_ptr += 1
                else:
                    row = ref_fake.iloc[fake_ptr % len(ref_fake)]
                    fake_ptr += 1
            
            assignments.append({
                'idx': len(assignments),
                'model_id': model_id,
                'prompt_type': pt,
                'source_title': row['title'],
                'source_text': row['text'] if tmpl['needs_text'] else '',
                'ai_humanal_label': row['ai_humanal_label'],
            })

df_plan = pd.DataFrame(assignments)

# Preview
print(f'\n\u2705 Total assignments: {len(df_plan)}')
print(f'\nPer model:')
print(df_plan['model_id'].value_counts())
print(f'\nPer prompt type:')
print(df_plan['prompt_type'].value_counts())

# Expected label distribution
df_plan['expected_label'] = df_plan.apply(
    lambda r: determine_fake_label(r['ai_humanal_label'], r['prompt_type']), axis=1
)
print(f'\nExpected fake_news_label distribution:')
print(f'  Real (0): {(df_plan["expected_label"]==0).sum()}')
print(f'  Fake (1): {(df_plan["expected_label"]==1).sum()}')


Referensi Real: 1500, Fake: 1500

✅ Total assignments: 6000

Per model:
model_id
gemma-4-31b-it    6000
Name: count, dtype: int64

Per prompt type:
prompt_type
paraphrase            1200
fabricate             1200
style_manipulation    1200
formal_fake           1200
emotional_real        1200
Name: count, dtype: int64

Expected fake_news_label distribution:
  Real (0): 3000
  Fake (1): 3000


## 8. Quick Test (1 Artikel per Skenario)

Test dulu 5 artikel sebelum jalankan batch 6.000.

In [11]:
print('Testing 1 artikel per skenario...')
print('='*60)

for pt in PROMPT_TYPES:
    row = df_plan[df_plan['prompt_type'] == pt].iloc[0]
    result = generate_article(
        title=row['source_title'],
        text=row['source_text'],
        prompt_type=row['prompt_type'],
        model_id=MODELS[0],  # Test dengan model pertama saja
    )
    label = determine_fake_label(row['ai_humanal_label'], row['prompt_type'])
    
    print(f'\n--- {pt.upper()} ---')
    print(f'Source title: {str(row["source_title"])[:80]}')
    print(f'AI/Humanal label: {row["ai_humanal_label"]} | Expected fake_news_label: {label} (ai_label = 1)')
    if result:
        print(f'Generated ({len(result)} chars): {result[:200]}...')
    else:
        print('FAILED!')
    
    time.sleep(3)  # Delay antar test

print(f'\n{"="*60}')
print('\u2705 Quick test selesai. Jika output di atas terlihat bagus,')
print('   lanjut ke Cell 9 untuk batch generation 6.000 artikel.')


Testing 1 artikel per skenario...

--- PARAPHRASE ---
Source title: Sean Spicer Confirms WH ‘Meeting with Potential People’ for Press Vacancies - Br
Original label: 0 | Expected fake_news_label: 0 (ai_label = 1)
Generated (1480 chars): **White House Evaluating Candidates for Communications Vacancies, Spicer Confirms**

WASHINGTON, D.C. — White House Press Secretary Sean Spicer announced during Tuesday's press briefing that the admin...

--- FABRICATE ---
Source title: The Future of Not Working - The New York Times
Original label: 0 | Expected fake_news_label: 1 (ai_label = 1)
Generated (1200 chars): The U.S. Department of Labor, in partnership with a consortium of leading artificial intelligence firms, announced on Tuesday the launch of the National Automation Transition Initiative (NATI), a swee...

--- STYLE_MANIPULATION ---
Source title: Ahmad Khan Rahami’s YouTube Account Listed Jihad Videos, Complaint Says - The Ne
Original label: 0 | Expected fake_news_label: 0 (ai_label = 1)
Gen

## 9. JALANKAN BATCH GENERATION! (Auto-Resume)

\u23f1\ufe0f Estimasi: **~3.5 jam** untuk 6.000 artikel.

- Progress auto-save setiap 50 artikel ke `ai_gen_progress.csv`
- Jika terputus, jalankan cell ini lagi \u2192 otomatis lanjut

In [15]:
# AUTO-RESUME
if os.path.exists(PROGRESS_CSV):
    df_progress = pd.read_csv(PROGRESS_CSV)
    completed_ids = set(df_progress['idx'].tolist())
    results = df_progress.to_dict('records')
    print(f'\u2705 Melanjutkan: {len(completed_ids)}/{len(df_plan)} selesai')
else:
    completed_ids = set()
    results = []
    print('\U0001f680 Memulai dari awal...')

total = len(df_plan)
failed_count = 0
start_time = time.time()

for i, row in df_plan.iterrows():
    if row['idx'] in completed_ids:
        continue
    
    generated_text = generate_article(
        title=row['source_title'],
        text=row['source_text'],
        prompt_type=row['prompt_type'],
        model_id=row['model_id']
    )
    
    if generated_text is None:
        failed_count += 1
        print(f'  \u274c #{row["idx"]} gagal')
        continue
    
    fake_label = determine_fake_label(row['ai_humanal_label'], row['prompt_type'])
    
    results.append({
        'idx': row['idx'],
        'title': row['source_title'],
        'text': generated_text,
        'fake_news_label': fake_label,
        'ai_label': 1,
        'source_model': row['model_id'],
        'prompt_type': row['prompt_type'],
    })
    completed_ids.add(row['idx'])
    
    done = len(completed_ids)
    remaining = total - done
    eta_min = remaining * 2.5 / 60
    
    if done % 10 == 0:
        print(f'[{done}/{total}] {row["prompt_type"]:20s} | '
              f'Model: {row["model_id"].split("-")[1]}... | '
              f'ETA: {eta_min:.0f}min | Failed: {failed_count}')
    
    if done % 50 == 0:
        pd.DataFrame(results).to_csv(PROGRESS_CSV, index=False)
        print(f'  \U0001f4be Saved ({done}/{total})')
    
    time.sleep(4.0)  # Safe delay untuk Gemini gratis (15 RPM)

# FINAL SAVE
df_ai = pd.DataFrame(results)
df_ai.to_csv(OUTPUT_CSV, index=False)

hrs = (time.time() - start_time) / 3600
print(f'\n{"="*60}')
print(f'\U0001f389 SELESAI! Artikel: {len(df_ai)} | Failed: {failed_count} | Waktu: {hrs:.1f} jam')
print(f'Tersimpan di: {OUTPUT_CSV}')


🚀 Memulai dari awal...


KeyboardInterrupt: 

## 10. Verifikasi Hasil

In [13]:
df_ai = pd.read_csv(OUTPUT_CSV)
print(f'Total AI articles: {len(df_ai)}')
print(f'\nPer model:')
print(df_ai['source_model'].value_counts())
print(f'\nPer prompt type:')
print(df_ai['prompt_type'].value_counts())
print(f'\nfake_news_label (0=Real, 1=Fake):')
print(df_ai['fake_news_label'].value_counts())
print(f'\nCross-tab model x prompt x label:')
print(df_ai.groupby(['source_model', 'prompt_type', 'fake_news_label']).size().unstack(fill_value=0))
print(f'\nText length:')
print(df_ai['text'].str.len().describe())

# Sample per skenario
print(f'\n{"="*60}')
print('SAMPLE PER SKENARIO:')
for pt in df_ai['prompt_type'].unique():
    row = df_ai[df_ai['prompt_type'] == pt].iloc[0]
    print(f'\n--- {pt.upper()} ---')
    print(f'Title: {str(row["title"])[:80]}')
    print(f'Label: fake={row["fake_news_label"]}, ai={row["ai_label"]}')
    print(f'Text: {str(row["text"])[:200]}...')

Total AI articles: 5350

Per model:
source_model
gemma-4-31b-it             5250
llama-3.3-70b-versatile     100
Name: count, dtype: int64

Per prompt type:
prompt_type
style_manipulation    1138
formal_fake           1095
paraphrase            1076
emotional_real        1036
fabricate             1005
Name: count, dtype: int64

fake_news_label (0=Real, 1=Fake):
fake_news_label
0    2683
1    2667
Name: count, dtype: int64

Cross-tab model x prompt x label:
fake_news_label                                0     1
source_model            prompt_type                   
gemma-4-31b-it          emotional_real      1036     0
                        fabricate              0  1005
                        formal_fake            0  1095
                        paraphrase           697   279
                        style_manipulation   850   288
llama-3.3-70b-versatile paraphrase           100     0

Text length:
count    5350.000000
mean      945.170467
std       530.120849
min        51.000000


In [14]:
# Cleanup progress file
if os.path.exists(PROGRESS_CSV):
    os.remove(PROGRESS_CSV)
    print(f'\U0001f5d1\ufe0f Progress file dihapus.')

print(f'\n\u2705 Output files:')
print(f'  1. {CLEAN_CSV} - WELFake setelah cleaning ({len(df)} rows)')
print(f'  2. {OUTPUT_CSV} - AI-generated articles ({len(df_ai)} rows)')
print(f'\n\U0001f449 Lanjut ke Step 2: build_final_dataset.ipynb')

🗑️ Progress file dihapus.

✅ Output files:
  1. welfake_clean.csv - WELFake setelah cleaning (60941 rows)
  2. ai_generated_dataset_v2.csv - AI-generated articles (5350 rows)

👉 Lanjut ke Step 2: build_final_dataset.ipynb
